In [1]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import (classification_report, confusion_matrix, mean_absolute_error,
                             mean_squared_error)    

clf_df = pd.read_csv("classification_escalation.csv")
reg_df = pd.read_csv("resolution_time_regression.csv")
display(clf_df.head(), reg_df.head())
print(clf_df.shape, reg_df.shape)
print(clf_df.isna().sum())
print(reg_df.isna().sum())
print("----------------------------------------------------")

print("- So what we're doing is there are two prediction tasks that are built on synthetic support ticket data.")
print("- Classification:- predicting whether a tickett will be escalated or not(categorical).")
print("- Regression:- predict resolution_time_hours(numerical).")
print("- We have excluded ticket_id because it is just an identifier.")



,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


(900, 7) (800, 7)
ticket_id                  0
complexity_score          22
team_experience_years     36
priority                  13
channel                    9
customer_tenure_months    31
escalated                  0
dtype: int64
ticket_id                  0
complexity_score          16
team_experience_years     24
priority                   0
channel                    0
customer_tenure_months    20
resolution_time_hours      0
dtype: int64
----------------------------------------------------
- So what we're doing is there are two prediction tasks that are built on synthetic support ticket data.
- Classification:- predicting whether a tickett will be escalated or not(categorical).
- Regression:- predict resolution_time_hours(numerical).
- We have excluded ticket_id because it is just an identifier.


In [2]:
num_cols = ["complexity_score", "team_experience_years", "customer_tenure_months"]
cat_cols = ["priority", "channel"]

def make_prep():     
    numeric = Pipeline([("impute", SimpleImputer(strategy="median")),
                        ("scale", StandardScaler())])
    categorical = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                            ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", numeric, num_cols), ("cat", categorical, cat_cols)])
print("----------------------------------------------------")

print("- Function is defined but not imputed before split in order to avoid leakage.")
print("- For the numeric columns we imputed using median cause it helps with outliers and then standardized for K-NN and Ridge penalty.")
print("- For categorical columns we used most-frequent imputation and one-hot encoding.")
print("- Everything is inside a pipeline.")

----------------------------------------------------
- Function is defined but not imputed before split in order to avoid leakage.
- For the numeric columns we imputed using median cause it helps with outliers and then standardized for K-NN and Ridge penalty.
- For categorical columns we used most-frequent imputation and one-hot encoding.
- Everything is inside a pipeline.


In [3]:
X_clf = clf_df.drop(columns=["ticket_id", "escalated"])   
y_clf = clf_df["escalated"]
print(y_clf.value_counts(normalize=True).round(3))

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=42, stratify=y_clf)
print(Xc_train.shape, Xc_test.shape, round(yc_train.mean(), 3), round(yc_test.mean(), 3))
print("----------------------------------------------------")

print("- The data is split into trainig and test set. 20% of the data is held out for the test set and it is not touched until the end.")

escalated
0    0.788
1    0.212
Name: proportion, dtype: float64
(720, 5) (180, 5) 0.212 0.211
----------------------------------------------------
- The data is split into trainig and test set. 20% of the data is held out for the test set and it is not touched until the end.


In [4]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   
clf_models = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "k-NN": KNeighborsClassifier(),
    "Decision tree": DecisionTreeClassifier(random_state=42),
}
rows = []
for name, model in clf_models.items():
    pipe = Pipeline([("prep", make_prep()), ("model", model)])
    r = cross_validate(pipe, Xc_train, yc_train, cv=skf,
                       scoring={"f1": "f1", "precision": "precision", "recall": "recall"})
    rows.append({"Model": name,
                 "F1 mean": r["test_f1"].mean(), "F1 std": r["test_f1"].std(),
                 "Precision mean": r["test_precision"].mean(), "Recall mean": r["test_recall"].mean()})
cv_clf = pd.DataFrame(rows)
display(cv_clf.round(3))
print("----------------------------------------------------")

print("- The models are compared with cross-validation(5-fold) on the trainig set and we use stratifiedkfold to get the same shuffled folds for every model.")
print("- Logistic regression is best beacuse it's F1 is higher compared to the others.")
print("- And it's chosen as the main matrix because the classes are not balanced.")

,Model,F1 mean,F1 std,Precision mean,Recall mean
0,Logistic regression,0.393,0.063,0.624,0.288
1,k-NN,0.361,0.050,0.562,0.268
2,Decision tree,0.331,0.058,0.318,0.347


----------------------------------------------------
- The models are compared with cross-validation(5-fold) on the trainig set and we use stratifiedkfold to get the same shuffled folds for every model.
- Logistic regression is best beacuse it's F1 is higher compared to the others.
- And it's chosen as the main matrix because the classes are not balanced.


In [5]:
param_grids = {
    "Logistic regression": {"model__C": [0.01, 0.1, 1, 10, 100]},
    "k-NN": {"model__n_neighbors": [3, 5, 9, 15, 21, 31]},
    "Decision tree": {"model__max_depth": [2, 3, 4, 5, 6, 8], "model__min_samples_leaf": [1, 5, 10, 20]},
}
tuned = {}
rows = []
for name, model in clf_models.items():
    pipe = Pipeline([("prep", make_prep()), ("model", model)])
    gs = GridSearchCV(pipe, param_grids[name], scoring="f1", cv=skf)
    gs.fit(Xc_train, yc_train)
    tuned[name] = gs
    rows.append({"Model": name, "Best params": gs.best_params_,
                 "Tuned CV F1 mean": gs.best_score_,
                 "Tuned CV F1 std": gs.cv_results_["std_test_score"][gs.best_index_]})
tuned_clf = pd.DataFrame(rows)
display(tuned_clf.round(3))

best_name = tuned_clf.loc[tuned_clf["Tuned CV F1 mean"].idxmax(), "Model"]
final_clf = tuned[best_name].best_estimator_
print("Selected:", best_name)
print("----------------------------------------------------")

print("- GridSearchCV helps us tune. it means that it chooses the best value for the modle's hyperparameter.")
print("- These hyperparameters are chosen on the cross-validation only.")
print("- For C in logistic regression it helps us control regularization so if it's too small it's strong but there's a risk of underfitting so C=10 was picked with median imputation.")
print("- For K-NN this helps us find the k most clossest training tickets small k would overfit because it follows noise and if it had been a large number it would have been too smooth and underfitted so GridSearchCV picked k=3 ")
print("- For decision tree max_depth and min_sample_leaf 6 and 20 chosen respectively. the deeper the tree goes it goes to overfitting.")

,Model,Best params,Tuned CV F1 mean,Tuned CV F1 std
0,Logistic regression,{'model__C': 10},0.423,0.088
1,k-NN,{'model__n_neighbors': 3},0.391,0.064
2,Decision tree,"{'model__max_depth': 6, 'model__min_samples_le...",0.381,0.080


Selected: Logistic regression
----------------------------------------------------
- GridSearchCV helps us tune. it means that it chooses the best value for the modle's hyperparameter.
- These hyperparameters are chosen on the cross-validation only.
- For C in logistic regression it helps us control regularization so if it's too small it's strong but there's a risk of underfitting so C=10 was picked with median imputation.
- For K-NN this helps us find the k most clossest training tickets small k would overfit because it follows noise and if it had been a large number it would have been too smooth and underfitted so GridSearchCV picked k=3 
- For decision tree max_depth and min_sample_leaf 6 and 20 chosen respectively. the deeper the tree goes it goes to overfitting.


In [6]:
pred = final_clf.predict(Xc_test)
print(confusion_matrix(yc_test, pred))
print(classification_report(yc_test, pred, zero_division=0))
tn, fp, fn, tp = confusion_matrix(yc_test, pred).ravel()
print("TN", tn, "FP", fp, "FN", fn, "TP", tp)
print("----------------------------------------------------")

print("- Since the selected model is logistic regression. on the test set F1 was 0.31 for the escalated tickets.")
print("- From the 38 escalated there was 30 false negative and it only causght 8. and there were 5 false positives. so what we observe is that when the model flaggs a ticket it is ususally correct but misses most escalations.")
print("- So the cost depends, if it is missing escalation then the better way is to lower threshold.")

[[137   5]
 [ 30   8]]
              precision    recall  f1-score   support

           0       0.82      0.96      0.89       142
           1       0.62      0.21      0.31        38

    accuracy                           0.81       180
   macro avg       0.72      0.59      0.60       180
weighted avg       0.78      0.81      0.77       180

TN 137 FP 5 FN 30 TP 8
----------------------------------------------------
- Since the selected model is logistic regression. on the test set F1 was 0.31 for the escalated tickets.
- From the 38 escalated there was 30 false negative and it only causght 8. and there were 5 false positives. so what we observe is that when the model flaggs a ticket it is ususally correct but misses most escalations.
- So the cost depends, if it is missing escalation then the better way is to lower threshold.


In [7]:
X_reg = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])
y_reg = reg_df["resolution_time_hours"]
print(y_reg.describe().round(2))
print("rows at exactly 0.5:", (y_reg == 0.5).sum(), "| above 15:", (y_reg > 15).sum())

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.20, random_state=42)
print(Xr_train.shape, Xr_test.shape)
print("----------------------------------------------------")

print("- Data is split into training and test set. no stratify because we use it for classification not here.")

count    800.00
mean       5.87
std        4.14
min        0.50
25%        2.63
50%        5.39
75%        8.35
max       27.49
Name: resolution_time_hours, dtype: float64
rows at exactly 0.5: 72 | above 15: 15
(640, 5) (160, 5)
----------------------------------------------------
- Data is split into training and test set. no stratify because we use it for classification not here.


In [8]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
reg_models = {
    "Linear regression": LinearRegression(),
    "Ridge (alpha=1)": Ridge(alpha=1.0),
    "Decision tree": DecisionTreeRegressor(random_state=42),
}
rows = []
for name, model in reg_models.items():
    pipe = Pipeline([("prep", make_prep()), ("model", model)])
    r = cross_validate(pipe, Xr_train, yr_train, cv=kf,
                       scoring={"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error"})
    rows.append({"Model": name, "CV MAE mean": -r["test_mae"].mean(), "CV MAE std": r["test_mae"].std(),
                 "CV RMSE mean": -r["test_rmse"].mean(), "CV RMSE std": r["test_rmse"].std()})
cv_reg = pd.DataFrame(rows)
display(cv_reg.round(3))
print("----------------------------------------------------")

print("- Same procedure as the classification but with no stratification and we add error metrics.")
print("- We compute MAE and RMSE. linear regression and Ridge have the same rsult since it's a small number of feature there is nothing to fix.")
print("- Decision tree is the worst(overfits).")
print("- For all the model there is a gap between MAE and RMSE this shows that there are signs of outliers.")

,Model,CV MAE mean,CV MAE std,CV RMSE mean,CV RMSE std
0,Linear regression,2.035,0.172,2.961,0.584
1,Ridge (alpha=1),2.035,0.171,2.960,0.583
2,Decision tree,3.235,0.055,4.582,0.379


----------------------------------------------------
- Same procedure as the classification but with no stratification and we add error metrics.
- We compute MAE and RMSE. linear regression and Ridge have the same rsult since it's a small number of feature there is nothing to fix.
- Decision tree is the worst(overfits).
- For all the model there is a gap between MAE and RMSE this shows that there are signs of outliers.


In [9]:
reg_grids = {
    "Ridge": {"model__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
    "Decision tree": {"model__max_depth": [2, 3, 4, 5, 6, 8], "model__min_samples_leaf": [1, 5, 10, 20]},
}
tuned_reg = {}
rows = []
for name, model in [("Ridge", Ridge()), ("Decision tree", DecisionTreeRegressor(random_state=42))]:
    pipe = Pipeline([("prep", make_prep()), ("model", model)])
    gs = GridSearchCV(pipe, reg_grids[name], scoring="neg_mean_absolute_error", cv=kf)
    gs.fit(Xr_train, yr_train)
    tuned_reg[name] = gs
    rows.append({"Model": name, "Best params": gs.best_params_,
                 "Tuned CV MAE mean": -gs.best_score_,
                 "Tuned CV MAE std": gs.cv_results_["std_test_score"][gs.best_index_]})
tuned_reg_df = pd.DataFrame(rows)
display(tuned_reg_df.round(3))

best_reg_name = tuned_reg_df.sort_values("Tuned CV MAE mean").iloc[0]["Model"]
final_reg = tuned_reg[best_reg_name].best_estimator_
print("Selected:", best_reg_name)
print("----------------------------------------------------")

print("- The same gridsearchCV concept applies here with MAE as the score.")
print("- Ridge 6 alpha values and the tree with 24 depth and leaf combinations.")

,Model,Best params,Tuned CV MAE mean,Tuned CV MAE std
0,Ridge,{'model__alpha': 0.1},2.035,0.172
1,Decision tree,"{'model__max_depth': 6, 'model__min_samples_le...",2.364,0.089


Selected: Ridge
----------------------------------------------------
- The same gridsearchCV concept applies here with MAE as the score.
- Ridge 6 alpha values and the tree with 24 depth and leaf combinations.


In [10]:
pr = final_reg.predict(Xr_test)
print("Test MAE:", round(mean_absolute_error(yr_test, pr), 3),
      "| Test RMSE:", round(np.sqrt(mean_squared_error(yr_test, pr)), 3))
print("----------------------------------------------------")

print("- Finally it's tested on the test set and the predictions are off by 2.2 hours on the mean.")
print("- The RMSE is larger because as the errors grow more they will count more and MAE misses count once.due to this they might prefer different models.")
print("- The test MAE is higher than the CV MAE due to the size of the split.")

Test MAE: 2.23 | Test RMSE: 3.403
----------------------------------------------------
- Finally it's tested on the test set and the predictions are off by 2.2 hours on the mean.
- The RMSE is larger because as the errors grow more they will count more and MAE misses count once.due to this they might prefer different models.
- The test MAE is higher than the CV MAE due to the size of the split.


In [11]:
results = pd.DataFrame({"actual": yr_test, "predicted": pr})
results["abs_error"] = (results["actual"] - results["predicted"]).abs()
display(results.sort_values("abs_error", ascending=False).head(5).round(2))

sq = ((results["actual"] - results["predicted"]) ** 2).sort_values(ascending=False)
print("Share of total squared error from the 5 biggest misses:", round(sq.head(5).sum() / sq.sum(), 3))
print("----------------------------------------------------")

print("- Most tickets are resolved about 11 hours.and a few of them the long ones are at the floor of 0.5 hours.if we see the three biggest misses on the test test they are 12 to 21 hours off.the resolution is right-skewed and the model cannot predict extreme tickets.")

,actual,predicted,abs_error
398,27.49,6.38,21.11
49,24.11,4.84,19.27
537,15.58,3.36,12.22
490,5.11,12.25,7.14
306,5.31,10.81,5.50


Share of total squared error from the 5 biggest misses: 0.565
----------------------------------------------------
- Most tickets are resolved about 11 hours.and a few of them the long ones are at the floor of 0.5 hours.if we see the three biggest misses on the test test they are 12 to 21 hours off.the resolution is right-skewed and the model cannot predict extreme tickets.


In [12]:
print("Reflection questions")
print("----------------------------------------------------")

print("1) Stratifying is used to make the class proportional and fair in the train and test set. a random split would put more or few escalations iin the test set which would make it unstable.")
print("2) All those things we did imputing, scaling are learned from the data we are not the once giving the value. as we have said earlier if we had done it before splitting then leakage would occur, which would lead to overfitting on the training set. so inside a pipeline, these things are evaluated on the training set only. and also it makes sure that the data follow the same step.")
print("3) Fold variability is to see how stable a score is. different data split gives us different results. so if the difference is smaller than the spread it should not be considered too much.")
print("4) Because if we do it more than once then the test result would go on and affect the decisions like choosing a model and soon on. which in turn woul dalso result in leakage. and the final score would end up being too optimistic.")
print("5) The speed of the prediction. plus the probabilities need to be calibrated-well. and also the cost of FN and FP. how the model handles missing values at prediction time.")

Reflection questions
----------------------------------------------------
1) Stratifying is used to make the class proportional and fair in the train and test set. a random split would put more or few escalations iin the test set which would make it unstable.
2) All those things we did imputing, scaling are learned from the data we are not the once giving the value. as we have said earlier if we had done it before splitting then leakage would occur, which would lead to overfitting on the training set. so inside a pipeline, these things are evaluated on the training set only. and also it makes sure that the data follow the same step.
3) Fold variability is to see how stable a score is. different data split gives us different results. so if the difference is smaller than the spread it should not be considered too much.
4) Because if we do it more than once then the test result would go on and affect the decisions like choosing a model and soon on. which in turn woul dalso result in leaka